In [ ]:
import queue
import time

import numpy as np
import tritonclient.grpc as grpcclient
import matplotlib.pyplot as plt
from IPython.display import Audio, display

TRITON_URL = "localhost:8001"
MODEL_NAME = "qwen3_tts"
SAMPLE_RATE = 24000

In [ ]:
def synthesize(text: str, language: str = "english") -> np.ndarray:
    """Send a TTS request and collect streamed audio chunks."""
    result_q: queue.Queue = queue.Queue()

    def _on_response(result, error):
        result_q.put((result, error))

    client = grpcclient.InferenceServerClient(url=TRITON_URL)
    client.start_stream(callback=_on_response)

    text_input = grpcclient.InferInput("text", [1, 1], "BYTES")
    text_input.set_data_from_numpy(np.array([[text]], dtype=object))

    lang_input = grpcclient.InferInput("language", [1, 1], "BYTES")
    lang_input.set_data_from_numpy(np.array([[language]], dtype=object))

    client.async_stream_infer(
        model_name=MODEL_NAME,
        inputs=[text_input, lang_input],
        outputs=[grpcclient.InferRequestedOutput("audio")],
    )

    chunks = []
    t0 = time.perf_counter()
    t_first = None

    while True:
        result, error = result_q.get(timeout=120)
        if error:
            client.stop_stream()
            raise RuntimeError(str(error))

        audio = result.as_numpy("audio").squeeze()
        if audio.size > 0:
            if t_first is None:
                t_first = time.perf_counter()
            chunks.append(audio)

        response = result.get_response()
        final_param = response.parameters.get("triton_final_response")
        if final_param and getattr(final_param, "bool_param", False):
            break

    client.stop_stream()

    elapsed = time.perf_counter() - t0
    ttfa = (t_first - t0) if t_first else elapsed
    print(f"Received {len(chunks)} chunks | TTFA: {ttfa:.3f}s | total: {elapsed:.3f}s")
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)

In [ ]:
import threading
import uuid


def synthesize_word_by_word(
    text: str,
    language: str = "english",
    inter_word_delay_s: float = 0.0,
) -> np.ndarray:
    """Feed transcript word-by-word using the server's multi-request text stream.

    When ``streaming`` is true and ``stream_id`` is set, ``model.py`` buffers
    ``text`` until ``end_of_text``; intermediate RPCs return empty audio (ack).
    The final RPC runs synthesis with incremental (HF-style) text conditioning.
    """
    words = text.split()
    if not words:
        return np.array([], dtype=np.float32)

    by_rid: dict[str, queue.Queue] = {}
    lock = threading.Lock()
    session_id = uuid.uuid4().hex

    def _on_response(result, error):
        rid = result.get_response().id
        if isinstance(rid, bytes):
            rid = rid.decode("utf-8")
        with lock:
            q = by_rid.get(rid)
        if q is not None:
            q.put((result, error))

    client = grpcclient.InferenceServerClient(url=TRITON_URL)
    client.start_stream(callback=_on_response)

    def _infer_chunk(chunk: str, end_of_text: bool, request_id: str) -> list[np.ndarray]:
        rq: queue.Queue = queue.Queue()
        with lock:
            by_rid[request_id] = rq

        text_in = grpcclient.InferInput("text", [1, 1], "BYTES")
        text_in.set_data_from_numpy(np.array([[chunk]], dtype=object))
        lang_in = grpcclient.InferInput("language", [1, 1], "BYTES")
        lang_in.set_data_from_numpy(np.array([[language]], dtype=object))
        stream_in = grpcclient.InferInput("streaming", [1, 1], "BOOL")
        stream_in.set_data_from_numpy(np.array([[True]], dtype=bool))
        eot_in = grpcclient.InferInput("end_of_text", [1, 1], "BOOL")
        eot_in.set_data_from_numpy(np.array([[end_of_text]], dtype=bool))
        sid_in = grpcclient.InferInput("stream_id", [1, 1], "BYTES")
        sid_in.set_data_from_numpy(np.array([[session_id]], dtype=object))

        client.async_stream_infer(
            model_name=MODEL_NAME,
            inputs=[text_in, lang_in, stream_in, eot_in, sid_in],
            outputs=[grpcclient.InferRequestedOutput("audio")],
            request_id=request_id,
        )

        out: list[np.ndarray] = []
        try:
            while True:
                result, error = rq.get(timeout=300)
                if error:
                    raise RuntimeError(str(error))
                audio = result.as_numpy("audio").squeeze()
                if audio.size > 0:
                    out.append(audio)
                resp = result.get_response()
                final = resp.parameters.get("triton_final_response")
                if final and getattr(final, "bool_param", False):
                    break
        finally:
            with lock:
                by_rid.pop(request_id, None)
        return out

    all_audio: list[np.ndarray] = []
    t0 = time.perf_counter()
    try:
        for i, w in enumerate(words):
            piece = w + (" " if i < len(words) - 1 else "")
            eot = i == len(words) - 1
            rid = f"w{i}-{uuid.uuid4().hex[:8]}"
            all_audio.extend(_infer_chunk(piece, eot, rid))
            if inter_word_delay_s > 0 and not eot:
                time.sleep(inter_word_delay_s)
    finally:
        client.stop_stream()

    elapsed = time.perf_counter() - t0
    print(
        f"Word-by-word: {len(words)} text RPCs | {len(all_audio)} audio segments | "
        f"wall {elapsed:.3f}s"
    )
    return np.concatenate(all_audio) if all_audio else np.array([], dtype=np.float32)


audio_stream = synthesize_word_by_word(
    "Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows.",
    language="english",
    inter_word_delay_s=0.05,
)
print(
    f"Stream path: {len(audio_stream)} samples — {len(audio_stream) / SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz"
)


In [ ]:
audio = synthesize(
    text="Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows.",
    language="english",
)
print(f"Got {len(audio)} samples — {len(audio) / SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz")

In [ ]:
t = np.arange(len(audio)) / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Qwen3-TTS Waveform")
fig.tight_layout()
plt.show()

display(Audio(audio, rate=SAMPLE_RATE))

In [ ]:
t = np.arange(len(audio_stream)) / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio_stream, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Qwen3-TTS streaming Waveform")
fig.tight_layout()
plt.show()

display(Audio(audio_stream, rate=SAMPLE_RATE))